# GDACS — Historical National Exposure

Retrieves country-level population exposure for all tropical cyclones within a date range, using the GDACS search endpoint for full historical coverage. Saves national level exposure estimates to blob storage for all available storms.

**API path:**
```
geteventlist/search  (paginated, filterable by date / source)
  -> getepisodedata  (latest episode per event)
    -> getimpact -> datums[alias='country'] -> ISO_3DIGIT, CNTRY_NAME, POP_AFFECTED
```

In [23]:
import requests
import pandas as pd
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

GDACS_BASE = 'https://www.gdacs.org/gdacsapi/api'
FROM_DATE = '2010-01-01'
TO_DATE   = '2026-12-31'
SOURCE    = 'NOAA'   # 'NOAA' = Atlantic/E.Pacific | 'JTWC' = W.Pacific/Indian Ocean
OUTPUT_CSV = 'gdacs_historical_national_exposure.csv'

# Wind speed (kt) implied by each buffer key
# Have spot-checked throughout, and there only ever seems to be these two buffers
BUFFER_KT = {
    'buffer39': 34,
    'buffer74': 64,
}

## 1. Fetch all TC events in the date range

In [24]:
# GDACS search API only returns Red/Orange storms by default
# Must search for each alert level separately to get all storms
all_events = []
ALERT_LEVELS = ['Red', 'Orange', 'Green']

for alert_level in ALERT_LEVELS:
    print(f'\nSearching for {alert_level} alert storms...')
    page = 1

    while True:
        params = {
            'eventlist':  'TC',
            'fromDate':   FROM_DATE,
            'toDate':     TO_DATE,
            'alertlevel': alert_level,
            'pageSize':   100,
            'pageNumber': page,
        }
        resp = requests.get(
            f'{GDACS_BASE}/events/geteventlist/search',
            params=params,
            timeout=30,
        )
        # API returns an empty body (not JSON) when there are no more pages
        if not resp.text.strip():
            break
        features = resp.json().get('features', [])
        if not features:
            break

        for f in features:
            p = f['properties']
            if SOURCE and p.get('source') != SOURCE:
                continue
            event_id = str(p['eventid'])
            # Avoid duplicates (some storms may change alert levels)
            if any(e['event_id'] == event_id for e in all_events):
                continue
            all_events.append({
                'event_id':    event_id,
                'storm_name':  p.get('name', ''),
                'source':      p.get('source', ''),
                'from_date':   p.get('fromdate', ''),
                'to_date':     p.get('todate', ''),
                'alert_level': p.get('alertlevel', ''),
            })

        print(f'  Page {page}: {len(features)} events fetched')
        page += 1

df_events = pd.DataFrame(all_events)
print(f'\nTotal TCs found ({SOURCE or "all basins"}): {len(df_events)}')


Searching for Red alert storms...
  Page 1: 100 events fetched
  Page 2: 21 events fetched

Searching for Orange alert storms...
  Page 1: 100 events fetched
  Page 2: 69 events fetched

Searching for Green alert storms...
  Page 1: 100 events fetched
  Page 2: 100 events fetched
  Page 3: 100 events fetched
  Page 4: 100 events fetched
  Page 5: 100 events fetched
  Page 6: 100 events fetched
  Page 7: 100 events fetched
  Page 8: 100 events fetched
  Page 9: 100 events fetched
  Page 10: 100 events fetched
  Page 11: 100 events fetched
  Page 12: 39 events fetched

Total TCs found (NOAA): 373


In [25]:
df_events.head()

,event_id,storm_name,source,from_date,to_date,alert_level
0,1001230,Tropical Cyclone MELISSA-25,NOAA,2025-10-21T15:00:00,2025-10-31T15:00:00,Red
1,1001168,Tropical Cyclone ERICK-25,NOAA,2025-06-16T21:00:00,2025-06-20T03:00:00,Red
2,1001101,Tropical Cyclone HELENE-24,NOAA,2024-09-23T15:00:00,2024-09-27T21:00:00,Red
3,1001067,Tropical Cyclone BERYL-24,NOAA,2024-06-28T21:00:00,2024-07-09T09:00:00,Red
4,1001028,Tropical Cyclone OTIS-23,NOAA,2023-10-22T15:00:00,2023-10-25T21:00:00,Red


## 2. For each event, fetch exposure data

From the last available episode.

In [ ]:
def fetch_national_exposure(event_id):
    '''Return country-row dicts for the latest episode. Empty list if no data.'''
    try:
        props = requests.get(
            f'{GDACS_BASE}/events/getepisodedata',
            params={'eventtype': 'TC', 'eventid': event_id},
            timeout=30,
        ).json()['properties']
    except Exception as e:
        print(f'  [{event_id}] failed: {e}')
        return []

    last_ep_url = props.get('episodes', [{}])[-1].get('details', '')
    episode_id  = last_ep_url.split('episodeid=')[-1].split('&')[0] if last_ep_url else '?'
    buffers     = {k: v for k, v in props.get('impacts', [{}])[0].get('resource', {}).items()
                   if k.startswith('buffer')}

    country_data = {}
    for buf, url in buffers.items():
        col = f'pop_{BUFFER_KT.get(buf, buf)}kt'
        try:
            datums = requests.get(url, timeout=30).json().get('datums', [])
        except Exception:
            continue
        country_datum = next((d for d in datums if d['alias'] == 'country'), None)
        if not country_datum:
            continue
        for row in country_datum.get('datum', []):
            sc   = {s['name']: s['value'] for s in row['scalars']['scalar']}
            iso3 = sc.get('ISO_3DIGIT')
            if not iso3:
                continue
            # Properly handle zeros vs missing data:
            # - If POP_AFFECTED exists (even if 0), store the value
            # - If POP_AFFECTED is missing or empty string, store None
            pop_affected = sc.get('POP_AFFECTED')
            if pop_affected is not None and pop_affected != '':
                # Convert to int, handling both string and numeric values
                try:
                    pop_value = int(float(pop_affected))
                except (ValueError, TypeError):
                    pop_value = None
            else:
                pop_value = None
            
            country_data.setdefault(iso3, {
                'event_id': event_id, 'episode_id': episode_id,
                'iso3': iso3, 'country_name': sc.get('CNTRY_NAME'),
            })[col] = pop_value

    return list(country_data.values())

In [27]:
all_rows = []

for _, ev in df_events.iterrows():
    eid  = ev['event_id']
    name = ev['storm_name']
    print(f'Fetching {name} ({eid}) …', end=' ')
    rows = fetch_national_exposure(eid)
    for r in rows:
        r['storm_name']  = name
        r['source']      = ev['source']
        r['from_date']   = ev['from_date']
        r['alert_level'] = ev['alert_level']
    all_rows.extend(rows)
    print(f'{len(rows)} countries')

df = (
    pd.DataFrame(all_rows)
    .sort_values(['from_date', 'storm_name'], ascending=False)
    .reset_index(drop=True)
)

# Reorder columns
leading  = ['storm_name', 'event_id', 'episode_id', 'source', 'from_date', 'alert_level',
            'iso3', 'country_name']
pop_cols = sorted([c for c in df.columns if c.startswith('pop_')])
df = df[leading + pop_cols]

Fetching Tropical Cyclone MELISSA-25 (1001230) … 9 countries
Fetching Tropical Cyclone ERICK-25 (1001168) … 1 countries
Fetching Tropical Cyclone HELENE-24 (1001101) … 1 countries
Fetching Tropical Cyclone BERYL-24 (1001067) … 13 countries
Fetching Tropical Cyclone OTIS-23 (1001028) … 1 countries
Fetching Tropical Cyclone NORMA-23 (1001024) … 1 countries
Fetching Tropical Cyclone LIDIA-23 (1001019) … 1 countries
Fetching Tropical Cyclone ROSLYN-22 (1000940) … 1 countries
Fetching Tropical Cyclone IAN-22 (1000923) … 2 countries
Fetching Tropical Cyclone FIONA-22 (1000916) … 18 countries
Fetching Tropical Cyclone IDA-21 (1000818) … 3 countries
Fetching Tropical Cyclone GRACE-21 (1000814) … 9 countries
Fetching Tropical Cyclone IOTA-20 (1000743) … 7 countries
Fetching Tropical Cyclone ETA-20 (1000738) … 7 countries
Fetching Tropical Cyclone DELTA-20 (1000726) … 3 countries
Fetching Tropical Cyclone MARIA-17 (1000404) … 0 countries
Fetching Tropical Cyclone IRMA-17 (1000393) … 11 countries

## 3. Clean results and join with IBTrACS

Get the IBTrACs data.

In [28]:
engine = stratus.get_engine("prod")
with engine.connect() as conn:
    df_ibtracs = pd.read_sql("SELECT * FROM storms.ibtracs_storms", conn)

Prepare the ADAM data to join, per-storm, with IBTrACS.

In [37]:
def parse_name(input_name):
    season_str = input_name.split("-")[-1]
    season = int("20" + season_str)
    storm_name = input_name.split("-")[0].split(" ")[-1]
    return season, storm_name

df_cleaned = df.copy()

df_cleaned[['season', 'name']] = df_cleaned["storm_name"].apply(
    lambda x: pd.Series(parse_name(x))
)

df_sel = df_cleaned.drop_duplicates(subset=["storm_name"]).drop(columns=["iso3", "country_name", "pop_34kt", "pop_64kt"]).reset_index()
print(f"Dataset has {len(df_sel)} unique storms.")

Dataset has 178 unique storms.


Join, per-storm, with the IBTrACS records.

In [39]:
df_merged = df_sel.merge(df_ibtracs, on=["season", "name"], how="left")
df_merged = df_merged.sort_values("provisional").drop_duplicates(subset="storm_name", keep="first")
assert(len(df_merged) == len(df_sel))

Now join back to the exposure records.

In [40]:
df_final_merged = df_cleaned.merge(df_merged)
assert(len(df_final_merged) == len(df_cleaned))

## 4. Save output results to Azure

In [41]:
stratus.upload_csv_to_blob(df_final_merged, "ds-cyclone-exposure/gdacs_historical_national_exposure.csv")